# 04 — A custom end-to-end analysis

Tie the pieces together: generate a dataset, then use the advanced
`run_scenario` helper to sweep a parameter and compare summary outcomes.

**Note:** runs the model once per scenario; compute-heavy.

In [ ]:
import sys
from pathlib import Path

# Reuse the advanced examples' helper (run_scenario + metric extraction).
sys.path.append(str(Path("../advanced").resolve()))
import _common

COUNTRY, LEVEL, START, END = "ETH", 2, 2015, 2017
base = _common.ensure_data(COUNTRY, LEVEL, START, END)

## Sweep R0 and compare outcomes

`run_scenario` builds and runs a SEIR model with parameter overrides and returns
summary metrics (peak timing/size, attack rate).

In [ ]:
import pandas as pd

rows = []
for r0 in [1.5, 2.5, 3.5]:
    result = _common.run_scenario(base, overrides={"r0": r0}, nyears=1)
    result["r0"] = r0
    rows.append(result)

results = pd.DataFrame(rows)[["r0", "attack_rate", "peak_day", "peak_infectious"]]
results

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(results.r0, results.attack_rate, marker="o")
ax1.set_xlabel("R0")
ax1.set_ylabel("attack rate")
ax1.set_title("Final size vs R0")
ax2.plot(results.r0, results.peak_day, marker="o", color="C1")
ax2.set_xlabel("R0")
ax2.set_ylabel("peak day")
ax2.set_title("Peak timing vs R0")
plt.tight_layout()

Higher R0 should raise the attack rate and pull the peak earlier. From here you
could calibrate to an observed attack rate (`../advanced/calibration_example.py`)
or parallelize a larger sweep (`../advanced/parallel_scenarios.py`).